In [21]:
import random 
import numpy as np
import glob
import cv2
from sklearn.utils import shuffle
from matplotlib import pyplot as plt
from sklearn.cluster import KMeans
import joblib

pssm_glob = 'psiblast/data/pssm/*.pssm'

In [20]:
def read_matrix_from_pssm_file(fname):
    nrows=0
    with open(fname) as txt_file:    # Count rows.
        for line in txt_file:
            l = line.strip().split()
            if (len(l) == 44):
                nrows+=1;
    matrix = np.zeros((nrows,20), dtype="int8")
    i=0
    with open(fname) as txt_file: # Read PSSM.
        for line in txt_file:
            l = line.strip().split()
            if (len(l) == 44):
                values = np.array(list(map(int, l[2:22])))
                matrix[i,:] = values
                i+=1
    return matrix

# LOAD PSSM Matrices into protein_descriptors
protein_descritors = {}
for file in glob.glob(pssm_glob):
    pssm_matrix = read_matrix_from_pssm_file(file)
    prot_name = file.split("\\")[1].replace(".pssm", "")
    protein_descritors[prot_name] = pssm_matrix 
    
print("PSSM Matrixes read:")
print(len(protein_descritors))
#joblib.dump(protein_descritors,"protein_descritors.pkl", compress=3)

PSSM Matrixes read:
0


In [22]:
protein_descritors = joblib.load("protein_descritors.pkl")

In [23]:
# Create expected outputs array from sec_strucure file DSSP
dataset_file = open("sec_structs.txt", "r")
protein_secundary_structure = {}
protein_aa = {}
x = 0
name = ""
for line in dataset_file:
    if (x%4==0):
        prot_name = line.strip()
    elif(x%4==1):
        protein_aa[prot_name] = line.strip()
    elif(x%4==3):
        protein_secundary_structure[prot_name] = line.strip()
    x+=1

proteins = list(protein_secundary_structure.keys())
print(len(proteins))

1461


In [17]:
# for each proteins, for each amino extracts PSSM window with AA centered.
from scipy.ndimage import shift

X_protein_features = {}

protein_index = 0
for protein in proteins:
    pssm = protein_descritors[protein]
    protein_features = np.zeros((len(pssm),20,13))

    translated_pssm = pssm.T
    offset_start=6
    #amino_class = []

    ix = 0
    for AA in range(0, len(pssm)):
        pssmmatrix = translated_pssm[:,AA+offset_start-6:AA+offset_start-6+13]
        pssmmatrix= shift(pssmmatrix, (0,offset_start), cval=0)
        if (len(pssmmatrix.T) <13):
            ncols = 13-len(pssmmatrix.T)
            aux = np.zeros((20,ncols), dtype="int8")
            pssmmatrix = np.hstack((pssmmatrix,aux))

        protein_features[ix] = pssmmatrix
        #amino_class.append(y_proteins[protein_index])
       
        if (offset_start >0):
            offset_start-=1
        ix+=1
   
    X_protein_features[protein] = protein_features
    protein_index +=1

In [24]:
print(len(proteins))
#remove proteins with wrong Structural information
for prot in proteins:
    if (len(X_protein_features[prot]) != len(protein_secundary_structure[prot])):
        protein_secundary_structure.pop(prot)
    if (len(X_protein_features[prot])<30):
        protein_secundary_structure.pop(prot)
        
proteins = list(protein_secundary_structure.keys())
print(len(proteins))


        

1461
1427


In [25]:
# for each proteins, for each amino extracts PSSM window with AA centered.
from scipy.ndimage import shift

X_protein_features = {}

protein_index = 0
for protein in proteins:
    pssm = protein_descritors[protein]
    protein_features = np.zeros((len(pssm),20,13))

    translated_pssm = pssm.T
    offset_start=6
    #amino_class = []

    ix = 0
    for AA in range(0, len(pssm)):
        pssmmatrix = translated_pssm[:,AA+offset_start-6:AA+offset_start-6+13]
        pssmmatrix= shift(pssmmatrix, (0,offset_start), cval=0)
        if (len(pssmmatrix.T) <13):
            ncols = 13-len(pssmmatrix.T)
            aux = np.zeros((20,ncols), dtype="int8")
            pssmmatrix = np.hstack((pssmmatrix,aux))

        protein_features[ix] = pssmmatrix
        #amino_class.append(y_proteins[protein_index])
       
        if (offset_start >0):
            offset_start-=1
        ix+=1
   
    X_protein_features[protein] = protein_features
    protein_index +=1

In [30]:
proteins = proteins[:100]

In [31]:
from sklearn.model_selection import train_test_split

proteins_train, proteins_test = train_test_split(proteins, test_size=0.30, random_state=42)


In [32]:
y_train = []
for prot_name in proteins_train:
    for S in protein_secundary_structure[prot_name]:
        y_train.append(S)
print(len(y_train))

y_test = []
for prot_name in proteins_test:
    for S in protein_secundary_structure[prot_name]:
        y_test.append(S)
print(len(y_test))

X_train = np.zeros((len(y_train), 20,13))
aa_ix = 0
for prot in proteins_train:
    for AA in X_protein_features[prot]:
        X_train[aa_ix] = AA
        aa_ix += 1

X_test = np.zeros((len(y_test), 20,13))
aa_ix = 0
for prot in proteins_test:
    for AA in X_protein_features[prot]:
        X_test[aa_ix] = AA
        aa_ix += 1
        
print(X_train.shape)
print(X_test.shape)

19296
4868
(19296, 20, 13)
(4868, 20, 13)


In [37]:
from sklearn import preprocessing
le = preprocessing.LabelEncoder()
le.fit(y_train)
y_train = le.transform(y_train)
y_test = le.transform(y_test)

In [38]:
import keras
INPUT_SHAPE = (20, 13, 1)  
inp = keras.layers.Input(shape=INPUT_SHAPE)
conv1 = keras.layers.Conv2D(500, kernel_size=(5, 5), activation='relu', padding='same')(inp)
pool1 = keras.layers.MaxPooling2D(pool_size=(2, 2))(conv1)
conv2 = keras.layers.Conv2D(100, kernel_size=(2, 2), activation='relu', padding='same')(pool1)
flat = keras.layers.Flatten()(conv2)  #Flatten the matrix to get it ready for dense.
hidden1 = keras.layers.Dense(50, activation='relu')(flat)
out = keras.layers.Dense(3, activation='softmax')(hidden1)   #units=1 gives error

model = keras.Model(inputs=inp, outputs=out)
model.compile(optimizer='adam',loss='categorical_crossentropy', metrics=['accuracy'])
print(model.summary())

Model: "model_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_2 (InputLayer)        [(None, 20, 13, 1)]       0         
                                                                 
 conv2d_2 (Conv2D)           (None, 20, 13, 500)       13000     
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 10, 6, 500)       0         
 2D)                                                             
                                                                 
 conv2d_3 (Conv2D)           (None, 10, 6, 100)        200100    
                                                                 
 flatten_1 (Flatten)         (None, 6000)              0         
                                                                 
 dense_2 (Dense)             (None, 50)                300050    
                                                           

In [39]:
from sklearn.model_selection import train_test_split
from keras.utils import to_categorical

#X_train, X_test, y_train, y_test = train_test_split(X, to_categorical(np.array(y)), test_size=0.33, random_state=42)

history = model.fit(X_train,
                         to_categorical(np.array(list(y_train))),
                         batch_size = 128,
                         verbose = 1,
                         epochs = 5,      #Changed to 3 from 50 for testing purposes.
                         validation_split = 0.2,
                         shuffle = False
                      #   callbacks=callbacks
                   )

print("Test_Accuracy: {:.2f}%".format(model.evaluate(X_test, to_categorical(np.array(list(y_test))) )[1]*100))


Epoch 1/5
121/121 [==============================] - 15s 118ms/step - loss: 0.9739 - accuracy: 0.5759 - val_loss: 0.7445 - val_accuracy: 0.7039
Epoch 2/5
121/121 [==============================] - 14s 115ms/step - loss: 0.7760 - accuracy: 0.6723 - val_loss: 0.6714 - val_accuracy: 0.7212
Epoch 3/5
121/121 [==============================] - 14s 116ms/step - loss: 0.6881 - accuracy: 0.7083 - val_loss: 0.6589 - val_accuracy: 0.7321
Epoch 4/5
121/121 [==============================] - 14s 115ms/step - loss: 0.6200 - accuracy: 0.7385 - val_loss: 0.6465 - val_accuracy: 0.7373
Epoch 5/5
153/153 [==============================] - 1s 9ms/step - loss: 0.7267 - accuracy: 0.7545
Test_Accuracy: 75.45%
